# Multilevel models

Multilevel models, also known as **hierarchical linear models (HLM)** or **mixed-effects models**, are statistical models designed for data that has a **nested or hierarchical structure**. This means observations are grouped within larger units.

For example:

  * Students nested within schools.
  * Patients nested within hospitals.
  * Repeated measurements nested within individuals.
  * Employees nested within departments within companies.

The key distinction of multilevel models from traditional regression (like OLS) is that they account for the **non-independence of observations within groups**. In traditional regression, it's assumed that all observations are independent. However, if students from the same school are more similar to each other than to students from other schools (e.g., due to shared teachers, curriculum, or school culture), this assumption is violated. Multilevel models address this by allowing for **variance at different levels of the hierarchy**.

-----

## Why Use Multilevel Models?

1.  **Correct Standard Errors and Inference**: Ignoring the nested structure can lead to underestimated standard errors, inflated Type I errors (false positives), and incorrect conclusions about statistical significance.
2.  **Partitioning Variance**: They allow us to determine how much of the variation in the outcome is at the individual level and how much is at the group level.
3.  **Modeling Group-Level Effects**: We can include variables that explain differences between groups (e.g., school funding, hospital size).
4.  **Modeling Varying Relationships**: We can examine if the relationship between an individual-level predictor and the outcome varies across groups (e.g., if the effect of study hours on test scores differs between schools).
5.  **Handling Unbalanced Data**: They can easily handle situations where groups have different numbers of observations.

-----


## Core Components: Fixed Effects and Random Effects

Multilevel models are often called "mixed-effects models" because they combine two types of effects:

  * **Fixed Effects**: These are the average effects of predictors across all groups. They represent the parameters that are constant across all units at a given level. In notation, these are typically denoted by $\\gamma$ (gamma).
  * **Random Effects**: These are the deviations of individual groups from the overall average (fixed effects). They account for the variability between groups that isn't explained by fixed predictors. Random effects are typically assumed to be normally distributed with a mean of zero and a certain variance. These are usually denoted by $u$ or $\\tau$ (tau).

-----

## Multilevel Regression (for Continuous Outcomes)

### Formula and Concepts

Let's consider a two-level model where individuals (level 1) are nested within groups (level 2).

**1. Random Intercept Model (Varying Intercepts)**

This is the simplest multilevel model. It assumes that the intercept (the baseline outcome) can vary across groups, but the effect of the level-1 predictor is consistent across all groups.

  * **Level 1 (Individual Level) Equation:**
    $$Y_{ij} = \beta_{0j} + \beta_{1j} X_{ij} + e_{ij}$$
    Where:

      * $Y\_{ij}$: Outcome for individual $i$ in group $j$.
      * $\\beta\_{0j}$: Intercept for group $j$ (this is what varies).
      * $\\beta\_{1j}$: Slope for predictor $X$ in group $j$. In a random intercept model, $\\beta\_{1j}$ is often fixed, i.e., $\\beta\_{1j} = \\gamma\_{10}$.
      * $X\_{ij}$: Level-1 predictor for individual $i$ in group $j$.
      * $e\_{ij}$: Level-1 residual (error) for individual $i$ in group $j$, assumed $e\_{ij} \\sim N(0, \\sigma^2\_e)$.

  * **Level 2 (Group Level) Equations:**
    $$\beta_{0j} = \gamma_{00} + u_{0j}$$   $$\beta_{1j} = \gamma_{10}$$ (if assuming a fixed slope)
    Where:

      * $\\gamma\_{00}$: Overall average intercept (fixed effect).
      * $u\_{0j}$: Random deviation of group $j$'s intercept from the overall average, assumed $u\_{0j} \\sim N(0, \\tau^2\_0)$.
      * $\\gamma\_{10}$: Overall average slope for $X$ (fixed effect).

  * **Combined (Mixed) Model:** Substitute the Level 2 equations into the Level 1 equation:
    $$Y_{ij} = (\gamma_{00} + u_{0j}) + \gamma_{10} X_{ij} + e_{ij}$$   $$Y_{ij} = \gamma_{00} + \gamma_{10} X_{ij} + u_{0j} + e_{ij}$$
    Here, $\\gamma\_{00}$ and $\\gamma\_{10}$ are fixed effects, and $u\_{0j}$ and $e\_{ij}$ are random effects, contributing to the overall error.

**2. Random Slope Model (Varying Intercepts and Slopes)**

This model allows both the intercept and the slope of a level-1 predictor to vary across groups.

  * **Level 1 (Individual Level) Equation:**
    $$Y_{ij} = \beta_{0j} + \beta_{1j} X_{ij} + e_{ij}$$

  * **Level 2 (Group Level) Equations:**
    $$\beta_{0j} = \gamma_{00} + u_{0j}$$   $$\beta_{1j} = \gamma_{10} + u_{1j}$$
    Where:

      * $u\_{1j}$: Random deviation of group $j$'s slope for $X$ from the overall average, assumed $(u\_{0j}, u\_{1j}) \\sim N(\\mathbf{0}, \\Sigma\_u)$, where $\\Sigma\_u$ is a covariance matrix representing the variances of $u\_{0j}$ and $u\_{1j}$ and their covariance.

  * **Combined (Mixed) Model:**
    $$Y_{ij} = (\gamma_{00} + u_{0j}) + (\gamma_{10} + u_{1j}) X_{ij} + e_{ij}$$   $$Y_{ij} = \gamma_{00} + \gamma_{10} X_{ij} + u_{0j} + u_{1j} X_{ij} + e_{ij}$$


### Example Data and Code (Regression)

Let's simulate data where student test scores are influenced by study hours and are nested within schools, with varying baseline scores (random intercept).


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Multilevel Regression Example ---")

# 1. Generate Synthetic Data
np.random.seed(42)
num_schools = 10
students_per_school = 30
total_students = num_schools * students_per_school

# School-level effects (random intercepts)
# Each school has a different "baseline" test score
school_intercepts = np.random.normal(loc=70, scale=5, size=num_schools)

# Fixed effect for study hours
fixed_slope_study_hours = 3

# Create DataFrame
data = []
for school_id in range(num_schools):
    for student_id_in_school in range(students_per_school):
        # Level 1: Student-level variables
        study_hours = np.random.uniform(low=1, high=10)
        # Level 2: School-level random intercept
        school_effect = school_intercepts[school_id]

        # Outcome variable: Test Score
        # TestScore = (Overall_Intercept) + (Fixed_Slope * StudyHours) + (School_Random_Intercept) + Error
        test_score = (
            fixed_slope_study_hours * study_hours
            + school_effect
            + np.random.normal(loc=0, scale=3) # individual error
        )
        data.append(
            {
                "SchoolID": school_id,
                "StudyHours": study_hours,
                "TestScore": test_score,
            }
        )

df = pd.DataFrame(data)
print("Sample of Synthetic Data:\n", df.head())
print(f"Total students: {len(df)}")
print(f"Number of schools: {df['SchoolID'].nunique()}")

# Visualize the data structure
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='StudyHours', y='TestScore', hue='SchoolID', palette='viridis', alpha=0.7)
plt.title('Test Scores vs. Study Hours by School (Simulated Data)')
plt.xlabel('Study Hours')
plt.ylabel('Test Score')
plt.legend(title='School ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 2. Fit a Multilevel Regression Model (Random Intercept)
# Formula syntax: outcome ~ fixed_effects + (random_effects_intercept | grouping_variable)
# Here, (1 | SchoolID) means a random intercept for each SchoolID
model = smf.mixedlm("TestScore ~ StudyHours", data=df, groups=df["SchoolID"])
result = model.fit()

print("\n--- Multilevel Regression Model Summary (Random Intercept) ---")
print(result.summary())

# Interpretation
# Fixed Effects:
# - Intercept: This is gamma_00, the average baseline test score when StudyHours is 0.
# - StudyHours: This is gamma_10, the average increase in test score for each additional hour of study,
#               across all schools.
# Random Effects:
# - Group Var: This is tau_0^2, the variance of the random intercepts (school_intercepts).
#              A significant value here suggests that schools indeed have different baseline test scores.
# - Level 1 Var (Residual): This is sigma_e^2, the variance of the individual-level errors.

# 3. Accessing Random Effects (e.g., estimated intercepts for each school)
print("\n--- Estimated Random Intercepts (Deviations from Fixed Intercept) ---")
# These are the estimated u_0j values
random_intercept_estimates = result.random_effects
for school_id, effect in random_intercept_estimates.items():
    print(f"School {school_id}: {effect['Intercept']:.2f}")

# You can add these deviations to the fixed intercept to get each school's estimated intercept
fixed_intercept = result.fe['Intercept']
print(f"\nOverall Fixed Intercept (gamma_00): {fixed_intercept:.2f}")
print("\n--- Estimated Intercept for Each School ---")
for school_id, effect in random_intercept_estimates.items():
    school_specific_intercept = fixed_intercept + effect['Intercept']
    print(f"School {school_id}: {school_specific_intercept:.2f}")


-----

## Multilevel Classification (Generalized Multilevel Models)

For classification, multilevel models are often referred to as **Generalized Linear Mixed Models (GLMMs)**, where the linear predictor is passed through a **link function** (e.g., logistic function for binary outcomes) to model the probability of an event.

### Formula and Concepts (Random Intercept Logistic Regression)

Let's consider a binary outcome ($Y\_{ij} \\in {0, 1}$) and a logit link function.

  * **Level 1 (Individual Level) Equation:**
    The probability of the outcome for individual $i$ in group $j$ is modeled using a logistic function:
    $$P(Y_{ij}=1 | \beta_{0j}, \beta_{1j}, X_{ij}) = \text{logit}^{-1}(\eta_{ij})$$
    where $\\text{logit}^{-1}(z) = \\frac{1}{1 + e^{-z}}$ and $\\eta\_{ij}$ is the linear predictor.
    The linear predictor is:
    $$\eta_{ij} = \beta_{0j} + \beta_{1j} X_{ij}$$

  * **Level 2 (Group Level) Equations:**
    Similar to regression, the intercept (and possibly slope) can vary:
    $$\beta_{0j} = \gamma_{00} + u_{0j}$$   $$\beta_{1j} = \gamma_{10}$$ (if assuming a fixed slope)

  * **Combined (Mixed) Model:**
    $$P(Y_{ij}=1 | \dots) = \text{logit}^{-1}(\gamma_{00} + \gamma_{10} X_{ij} + u_{0j})$$
    Here, $u\_{0j}$ represents the random effect on the log-odds scale.

### Example Data and Code (Classification)

Let's simulate data for student admission probability based on test scores, nested within schools, where schools have varying admission baselines.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("\n--- Multilevel Classification Example ---")

# 1. Generate Synthetic Data for Classification
np.random.seed(42)
num_schools = 10
students_per_school = 30
total_students = num_schools * students_per_school

# School-level effects (random intercepts on the log-odds scale)
school_baseline_log_odds = np.random.normal(loc=-1.0, scale=0.8, size=num_schools) # Average log-odds of admission

# Fixed effect for test score (on the log-odds scale)
fixed_slope_test_score = 0.5 # A higher test score increases log-odds of admission

# Create DataFrame
data_cls = []
for school_id in range(num_schools):
    for student_id_in_school in range(students_per_school):
        # Level 1: Student-level variables
        test_score = np.random.uniform(low=50, high=100) # Student's test score

        # Level 2: School-level random intercept
        school_effect_log_odds = school_baseline_log_odds[school_id]

        # Calculate linear predictor (on log-odds scale)
        linear_predictor = (
            fixed_slope_test_score * (test_score - 75) / 10 # Center and scale test score for better interpretation
            + school_effect_log_odds
        )

        # Convert log-odds to probability using logistic function
        probability_admission = 1 / (1 + np.exp(-linear_predictor))

        # Binary outcome: Admission (1) or Not Admitted (0)
        admission = 1 if np.random.rand() < probability_admission else 0

        data_cls.append(
            {
                "SchoolID": school_id,
                "TestScore": test_score,
                "Admission": admission,
            }
        )

df_cls = pd.DataFrame(data_cls)
print("Sample of Synthetic Classification Data:\n", df_cls.head())
print(f"Total students: {len(df_cls)}")
print(f"Number of schools: {df_cls['SchoolID'].nunique()}")
print(f"Admission counts:\n{df_cls['Admission'].value_counts()}")

# Visualize the data structure (TestScore vs. Admission probability for each school)
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_cls.groupby(['SchoolID', pd.cut(df_cls['TestScore'], bins=10)])['Admission'].mean().reset_index(),
             x='TestScore', y='Admission', hue='SchoolID', palette='viridis', marker='o', alpha=0.7)
plt.title('Admission Probability vs. Test Score by School (Simulated Data)')
plt.xlabel('Test Score')
plt.ylabel('Admission Probability')
plt.ylim(-0.1, 1.1)
plt.legend(title='School ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# 2. Fit a Multilevel Logistic Regression Model (Random Intercept)
# For GLMM, use 'smf.gee' or 'smf.glm' with a family and 'smf.mixedlm' for random effects
# statsmodels' mixedlm directly supports GLMMs with specific families.
# 'Binomial' family for binary outcomes, 'logit' as the default link function.
model_cls = smf.mixedlm(
    "Admission ~ TestScore",
    data=df_cls,
    groups=df_cls["SchoolID"],
    family=sm.families.Binomial()
)
result_cls = model_cls.fit()

print("\n--- Multilevel Classification Model Summary (Random Intercept Logistic) ---")
print(result_cls.summary())

# Interpretation
# Fixed Effects:
# - Intercept: Average log-odds of admission for a student with a TestScore of 0 (after centering/scaling if done).
# - TestScore: Average change in log-odds of admission for a one-unit increase in TestScore, across all schools.
# Random Effects:
# - Group Var: Variance of the random intercepts on the log-odds scale.
#              Indicates how much school baseline admission rates vary.

# 3. Make predictions and evaluate
# Predict probabilities on the original scale
df_cls['Predicted_Prob'] = result_cls.predict(df_cls)
df_cls['Predicted_Admission'] = (df_cls['Predicted_Prob'] > 0.5).astype(int) # Convert probabilities to binary class

print("\n--- Model Evaluation (Classification) ---")
print(f"Overall Accuracy: {accuracy_score(df_cls['Admission'], df_cls['Predicted_Admission']):.4f}")
print(f"Overall ROC AUC: {roc_auc_score(df_cls['Admission'], df_cls['Predicted_Prob']):.4f}")
print("Classification Report:\n", classification_report(df_cls['Admission'], df_cls['Predicted_Admission']))

# Example of predicting for a new student in an existing school (e.g., School 0)
new_student_school_0_data = pd.DataFrame({
    'SchoolID': [0],
    'TestScore': [85]
})
predicted_prob_new = result_cls.predict(new_student_school_0_data)
print(f"\nPredicted probability of admission for a student with TestScore 85 in School 0: {predicted_prob_new[0]:.4f}")

# Example of estimating a school-specific probability curve
plt.figure(figsize=(8, 6))
school_to_plot = 5 # Pick an arbitrary school
school_df = df_cls[df_cls['SchoolID'] == school_to_plot].copy()

# Generate x-values for plotting the curve
test_scores_range = np.linspace(school_df['TestScore'].min() - 5, school_df['TestScore'].max() + 5, 100)
temp_df_for_plot = pd.DataFrame({
    'SchoolID': school_to_plot,
    'TestScore': test_scores_range
})

# Predict probabilities for this school using the multilevel model
predicted_prob_curve = result_cls.predict(temp_df_for_plot)

sns.scatterplot(data=school_df, x='TestScore', y='Admission', hue='Admission', palette='coolwarm', s=100, alpha=0.7, legend=False)
plt.plot(test_scores_range, predicted_prob_curve, color='blue', linewidth=2, label=f'Predicted Prob. (School {school_to_plot})')
plt.title(f'Multilevel Logistic Regression for School {school_to_plot}')
plt.xlabel('Test Score')
plt.ylabel('Admission Probability')
plt.ylim(-0.1, 1.1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()



-----

### Understanding the Output and Interpretation

In both the regression and classification examples, the `statsmodels` library provides a summary that is rich with information:

  * **Fixed Effects**: These are the `coef` values for your predictors (e.g., `StudyHours` or `TestScore`) and the `Intercept`. They represent the average effect of these variables across all groups.
  * **Random Effects**: Look for `Group Var` (for `SchoolID` in our case) and `Residual` (or `Level 1 Var`).
      * **`Group Var` ($\\tau\_0^2$ for random intercept)**: This is the variance of the random intercepts. A statistically significant `Group Var` (check its standard error and p-value if available, or just the magnitude if it's large) indicates that there is significant variability in the intercepts between groups. In our test score example, this means schools have different baseline test scores that are not explained by the `StudyHours` alone.
      * **`Residual` ($\\sigma\_e^2$ for level-1 error)**: This is the variance of the errors at the individual level (what's left unexplained after accounting for fixed effects and random group intercepts).
  * **Intraclass Correlation Coefficient (ICC)**: While not directly in the `statsmodels` summary for `mixedlm` by default, it's a very important concept in multilevel models. For a random intercept model, ICC tells you the proportion of total variance in the outcome that is attributable to the group level.
    $$\text{ICC} = \frac{\tau_0^2}{\tau_0^2 + \sigma_e^2}$$
    A higher ICC (e.g., \> 0.10) suggests that a multilevel model is indeed appropriate because a significant portion of the variance is at the group level, violating the independence assumption of OLS.

Multilevel models are powerful tools for analyzing complex data structures and provide more accurate and nuanced insights into how phenomena vary across different levels of a hierarchy. While the formulas can look intimidating, the core idea is to disentangle variance components and account for clustering.